Ноутбук считает метрики для исходного PromptRetriever до русского дообучения. Это базовая точка сравнения для финальной модели.


In [ ]:
!pip -q install faiss-cpu peft==0.13.2 bitsandbytes==0.46.1 sentencepiece protobuf accelerate

import os, json, math, time, hashlib, gc
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn.functional as F
import faiss

from huggingface_hub import login, snapshot_download
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig
from peft import PeftModel


SOURCE_ENC_DIR = "/kaggle/input/datasets/sukiss/corpus-embedings"
METRIC_TESTSET = "/kaggle/input/datasets/sukiss/dataset-for-metrics/chunks_testset_metric_instructions.jsonl"

BASE_MODEL = "meta-llama/Llama-2-7b-hf"
PROMPTRIEVER_ADAPTER_REPO = "samaya-ai/promptriever-llama2-7b-v1"

ADAPTER_LOCAL = "/kaggle/working/promptriever_original_adapter"

OUT_DIR = "/kaggle/working/promptriever_original_encoding"
EVAL_OUT_DIR = "/kaggle/working/promptriever_original_metric_eval"

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(EVAL_OUT_DIR, exist_ok=True)

QUERY_MAX_LEN = 192
PASSAGE_MAX_LEN = 128
BATCH_DOCS = 16
BATCH_QUERIES = 32

HF_TOKEN = globals().get("HF_TOKEN", os.environ.get("HF_TOKEN", ""))
if HF_TOKEN:
    login(token=HF_TOKEN) if HF_TOKEN else None


if not os.path.exists(os.path.join(ADAPTER_LOCAL, "adapter_config.json")):
    snapshot_download(
        repo_id=PROMPTRIEVER_ADAPTER_REPO,
        local_dir=ADAPTER_LOCAL,
        token=HF_TOKEN if HF_TOKEN else None,
    )

print("Adapter files:", sorted(os.listdir(ADAPTER_LOCAL))[:30])


def sha1_text(t):
    return hashlib.sha1(t.strip().encode("utf-8")).hexdigest()

def load_docids(path):
    with open(path, "r", encoding="utf-8") as f:
        return [x.strip() for x in f if x.strip()]

def load_corpus_texts(source_dir, name):
    corpus_path = os.path.join(source_dir, f"{name}_corpus.jsonl")
    docids_path = os.path.join(source_dir, f"{name}_docids.txt")

    docids = load_docids(docids_path)
    text_by_docid = {}

    with open(corpus_path, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            text_by_docid[str(obj["docid"])] = obj["text"]

    texts = [text_by_docid[d] for d in docids]
    return docids, texts

def write_corpus_files(name, docids, texts):
    with open(os.path.join(OUT_DIR, f"{name}_docids.txt"), "w", encoding="utf-8") as f:
        for d in docids:
            f.write(d + "\n")

    with open(os.path.join(OUT_DIR, f"{name}_corpus.jsonl"), "w", encoding="utf-8") as f:
        for d, t in zip(docids, texts):
            f.write(json.dumps({"docid": d, "text": t}, ensure_ascii=False) + "\n")

def infer_dim_from_file(path, n_rows, dtype=np.float16):
    return os.path.getsize(path) // (n_rows * np.dtype(dtype).itemsize)

def mrr_at_k(rank, k):
    return 1.0 / rank if rank <= k else 0.0

def ndcg_at_k(rank, k):
    return 1.0 / math.log2(rank + 1) if rank <= k else 0.0

def ap_at_k(rank, k):
    return 1.0 / rank if rank <= k else 0.0

def hit_at_k(rank, k):
    return 1.0 if rank <= k else 0.0

def sicr_score(r_ori, s_ori, r_ins, s_ins, r_rev, s_rev):
    return float(
        (r_ins < r_ori) and
        (s_ins > s_ori) and
        (r_ori < r_rev) and
        (s_ori > s_rev)
    )

def wise_score(r_ori, r_ins, r_rev, n_positive_original=1, k=20):
    if r_ins <= r_ori < r_rev:
        if r_ori <= n_positive_original and r_ins == 1:
            return 1.0
        if r_ori <= k:
            return (1.0 - (math.sqrt(max(0, r_ori - r_ins)) / k)) * (1.0 / math.sqrt(r_ins))
        return 0.01

    if r_rev < r_ori < r_ins:
        return -1.0
    if r_ori <= r_ins:
        return (r_ori - r_ins) / r_ins
    if r_rev <= r_ori:
        return (r_rev - r_ori) / r_ori

    return 0.0

def pmrr_doc_score(rank_old, rank_new):
    mrr_old = 1.0 / rank_old
    mrr_new = 1.0 / rank_new

    if rank_old > rank_new:
        return (mrr_old / mrr_new) - 1.0
    return 1.0 - (mrr_new / mrr_old)


gc.collect()
torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

try:
    tok = AutoTokenizer.from_pretrained(ADAPTER_LOCAL, use_fast=False, token=HF_TOKEN if HF_TOKEN else None)
except Exception:
    tok = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=False, token=HF_TOKEN if HF_TOKEN else None)

if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "right"

base = AutoModel.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    token=HF_TOKEN if HF_TOKEN else None,
)
base.config.use_cache = False

model = PeftModel.from_pretrained(
    base,
    ADAPTER_LOCAL,
    is_trainable=False,
)

model.eval()

MODEL_DEVICE = next(model.parameters()).device
print("Loaded PromptRetriever original. device:", MODEL_DEVICE)

def add_eos(texts):
    return [t + tok.eos_token for t in texts]

def eos_pool(last_hidden_state, attention_mask):
    lengths = attention_mask.sum(dim=1) - 1
    idx = torch.arange(last_hidden_state.size(0), device=last_hidden_state.device)
    return last_hidden_state[idx, lengths]

@torch.inference_mode()
def encode_texts(texts, max_len):
    batch = tok(
        add_eos(texts),
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors="pt",
    )
    batch = {k: v.to(MODEL_DEVICE) for k, v in batch.items()}
    out = model(**batch)
    emb = eos_pool(out.last_hidden_state, batch["attention_mask"])
    emb = F.normalize(emb, p=2, dim=-1)
    return emb.detach().to(torch.float32).cpu().numpy()


def encode_corpus(name):
    out_emb = os.path.join(OUT_DIR, f"{name}_embeddings.npy")
    out_docids = os.path.join(OUT_DIR, f"{name}_docids.txt")
    out_corpus = os.path.join(OUT_DIR, f"{name}_corpus.jsonl")

    if os.path.exists(out_emb) and os.path.exists(out_docids) and os.path.exists(out_corpus):
        print(f"[{name}] embeddings already exist, skipping.")
        return

    docids_local, texts = load_corpus_texts(SOURCE_ENC_DIR, name)
    write_corpus_files(name, docids_local, texts)

    print(f"[{name}] encoding docs:", len(texts))

    first = encode_texts(texts[:1], PASSAGE_MAX_LEN).astype(np.float16)
    dim = first.shape[1]

    emb = np.memmap(out_emb, dtype=np.float16, mode="w+", shape=(len(texts), dim))
    emb[0:1] = first

    t0 = time.time()

    for i in range(1, len(texts), BATCH_DOCS):
        j = min(len(texts), i + BATCH_DOCS)
        emb[i:j] = encode_texts(texts[i:j], PASSAGE_MAX_LEN).astype(np.float16)

        if j % 500 == 0:
            print(f"[{name}] encoded {j}/{len(texts)}")

    emb.flush()
    print(f"[{name}] saved:", out_emb, "time_s:", round(time.time() - t0, 1))

encode_corpus("main")
encode_corpus("test")


def load_one_encoded(name):
    docids_local = load_docids(os.path.join(OUT_DIR, f"{name}_docids.txt"))
    emb_path = os.path.join(OUT_DIR, f"{name}_embeddings.npy")
    dim = infer_dim_from_file(emb_path, len(docids_local), dtype=np.float16)
    emb = np.memmap(emb_path, dtype=np.float16, mode="r", shape=(len(docids_local), dim))

    hashes = []
    with open(os.path.join(OUT_DIR, f"{name}_corpus.jsonl"), "r", encoding="utf-8") as f:
        for line in f:
            hashes.append(sha1_text(json.loads(line)["text"]))

    return docids_local, hashes, emb, dim

main_docids, main_hashes, main_emb, dim1 = load_one_encoded("main")
test_docids, test_hashes, test_emb, dim2 = load_one_encoded("test")
assert dim1 == dim2

docids = [f"main::{x}" for x in main_docids] + [f"test::{x}" for x in test_docids]
text_hashes = main_hashes + test_hashes

xb = np.vstack([
    np.asarray(main_emb, dtype=np.float32),
    np.asarray(test_emb, dtype=np.float32),
])

hash_to_indices = defaultdict(list)
for i, h in enumerate(text_hashes):
    hash_to_indices[h].append(i)

index = faiss.IndexFlatIP(dim1)
index.add(xb)

print("FAISS docs:", index.ntotal, "dim:", dim1)


def load_metric_items(path):
    items = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            it = json.loads(line)
            if all(it.get(k) for k in [
                "query_id", "query", "only_instruction",
                "reverse_instruction", "pmrr_instruction",
                "positive", "pmrr_changed_docs"
            ]):
                items.append(it)
    return items

def query_text(it, mode):
    q = it["query"].strip()
    if mode == "orig":
        return q
    if mode == "inst":
        return (q + " " + it["only_instruction"].strip()).strip()
    if mode == "rev":
        return (q + " " + it["reverse_instruction"].strip()).strip()
    if mode == "pmrr":
        return (q + " " + it["pmrr_instruction"].strip()).strip()
    raise ValueError(mode)

def encode_all_queries(texts):
    out = []
    for i in range(0, len(texts), BATCH_QUERIES):
        out.append(encode_texts(texts[i:i+BATCH_QUERIES], QUERY_MAX_LEN))
    return np.vstack(out)

def best_rank_score(I_row, D_row, candidate_indices):
    candidates = set(candidate_indices)
    for pos, idx in enumerate(I_row):
        if int(idx) in candidates:
            return pos + 1, float(D_row[pos])
    return len(I_row) + 1, float("-inf")

items = load_metric_items(METRIC_TESTSET)

kept = []
for it in items:
    if sha1_text(it["positive"]) in hash_to_indices:
        kept.append(it)

items = kept
print("Metric items kept:", len(items))

q_orig = encode_all_queries([query_text(x, "orig") for x in items])
q_inst = encode_all_queries([query_text(x, "inst") for x in items])
q_rev = encode_all_queries([query_text(x, "rev") for x in items])
q_pmrr = encode_all_queries([query_text(x, "pmrr") for x in items])

topk = len(docids)

D_orig, I_orig = index.search(q_orig, topk)
D_inst, I_inst = index.search(q_inst, topk)
D_rev, I_rev = index.search(q_rev, topk)
D_pmrr, I_pmrr = index.search(q_pmrr, topk)

rows = []
stats = Counter()

for i, it in enumerate(items):
    pos_candidates = hash_to_indices[sha1_text(it["positive"])]

    r_ori, s_ori = best_rank_score(I_orig[i], D_orig[i], pos_candidates)
    r_ins, s_ins = best_rank_score(I_inst[i], D_inst[i], pos_candidates)
    r_rev, s_rev = best_rank_score(I_rev[i], D_rev[i], pos_candidates)

    changed_scores = []

    for ch in it.get("pmrr_changed_docs", []):
        h = ch.get("text_hash") or sha1_text(ch.get("text", ""))
        cand = hash_to_indices.get(h)

        if not cand:
            stats["missing_pmrr_changed_doc"] += 1
            continue

        r_old, _ = best_rank_score(I_inst[i], D_inst[i], cand)
        r_new, _ = best_rank_score(I_pmrr[i], D_pmrr[i], cand)
        changed_scores.append(pmrr_doc_score(r_old, r_new))

    rows.append({
        "query_id": str(it["query_id"]),
        "instruction_style": it.get("instruction_style", "unknown"),

        "rank_orig": r_ori,
        "rank_inst": r_ins,
        "rank_rev": r_rev,

        "score_orig": s_ori,
        "score_inst": s_ins,
        "score_rev": s_rev,

        "sicr": sicr_score(r_ori, s_ori, r_ins, s_ins, r_rev, s_rev),
        "wise": wise_score(r_ori, r_ins, r_rev),
        "pmrr": float(np.mean(changed_scores)) if changed_scores else None,
    })

per_query_out = os.path.join(EVAL_OUT_DIR, "promptriever_original_metric_eval_combined_per_query.jsonl")

with open(per_query_out, "w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

def summarize_mode(rows, mode):
    ranks = [r[f"rank_{mode}"] for r in rows]
    return {
        "MRR@10": float(np.mean([mrr_at_k(r, 10) for r in ranks])),
        "nDCG@5": float(np.mean([ndcg_at_k(r, 5) for r in ranks])),
        "nDCG@10": float(np.mean([ndcg_at_k(r, 10) for r in ranks])),
        "MAP@1000": float(np.mean([ap_at_k(r, 1000) for r in ranks])),
        "Hit@10": float(np.mean([hit_at_k(r, 10) for r in ranks])),
    }

pmrr_values = [r["pmrr"] for r in rows if r["pmrr"] is not None]

summary = {
    "model": "PromptRetriever-original",
    "count": len(rows),
    "orig": summarize_mode(rows, "orig"),
    "inst": summarize_mode(rows, "inst"),
    "rev": summarize_mode(rows, "rev"),
    "instruction_metrics": {
        "SICR": float(np.mean([r["sicr"] for r in rows])),
        "SICR_x100": float(100 * np.mean([r["sicr"] for r in rows])),
        "WISE": float(np.mean([r["wise"] for r in rows])),
        "WISE_x100": float(100 * np.mean([r["wise"] for r in rows])),
        "pMRR": float(np.mean(pmrr_values)) if pmrr_values else None,
        "pMRR_x100": float(100 * np.mean(pmrr_values)) if pmrr_values else None,
    },
    "stats": dict(stats),
}

summary_out = os.path.join(EVAL_OUT_DIR, "promptriever_original_metric_eval_combined_summary.json")

with open(summary_out, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("DONE")
print("Per-query:", per_query_out)
print("Summary:", summary_out)
print(json.dumps(summary, ensure_ascii=False, indent=2))
